# Работа со строковыми значениями

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Работа со строковыми значениям"
* https://pyformat.info/
* https://docs.python.org/3/library/re.html
    * https://docs.python.org/3/library/re.html#flags
    * https://docs.python.org/3/library/re.html#functions
* https://pythonru.com/primery/primery-primeneniya-regulyarnyh-vyrazheniy-v-python
* https://kanoki.org/2019/11/12/how-to-use-regex-in-pandas/
* https://realpython.com/nltk-nlp-python/

## Задачи для совместного разбора

1. Вывести на экран данные из словаря `obj` построчно в виде `k = v`, задав формат таким образом, чтобы знак равенства оказался на одной и той же позиции во всех строках. Строковые литералы обернуть в кавычки.

In [ ]:
obj = {
    "home_page": "https://github.com/pypa/sampleproject",
    "keywords": "sample setuptools development",
    "license": "MIT",
}

2. Написать регулярное выражение,которое позволит найти номера групп студентов.

In [ ]:
obj = pd.Series(["Евгения гр.ПМ19-1", "Илья пм 20-4", "Анна 20-3"])
obj

0    Евгения гр.ПМ19-1
1         Илья пм 20-4
2            Анна 20-3
dtype: object

3. Разбейте текст формулировки задачи 2 на слова.

## Лабораторная работа 6

### Форматирование строк

In [ ]:
import pandas as pd
import numpy as np

1\. Загрузите данные из файла `recipes_sample.csv` (__ЛР2__) в виде `pd.DataFrame` `recipes` При помощи форматирования строк выведите информацию об id рецепта и времени выполнения 5 случайных рецептов в виде таблицы следующего вида:

    
    |      id      |  minutes  |
    |--------------------------|
    |    61178     |    65     |
    |    202352    |    80     |
    |    364322    |    150    |
    |    26177     |    20     |
    |    224785    |    35     |
    
Обратите внимание, что ширина столбцов заранее неизвестна и должна рассчитываться динамически, в зависимости от тех данных, которые были выбраны.

In [ ]:
recipes = pd.read_csv('recipes_sample.csv')
random_recipes = recipes[['id', 'minutes']].sample(5)

id_width = max(len(str(random_recipes['id'].max())), len('id'))
minutes_width = max(len(str(random_recipes['minutes'].max())), len('minutes'))

print(f"| {'id':^{id_width}} | {'minutes':^{minutes_width}} |")
print(f"|------------------|")

for index, row in random_recipes.iterrows():
    print(f"| {row['id']:^{id_width}} | {row['minutes']:^{minutes_width}} |")

|   id   | minutes |
|------------------|
| 269340 |   35    |
| 40440  |   35    |
| 202413 |    5    |
| 93778  |   150   |
| 57090  |   50    |


2\. Напишите функцию `show_info`, которая по данным о рецепте создает строку (в смысле объекта python) с описанием следующего вида:

```
"Название Из Нескольких Слов"

1. Шаг 1
2. Шаг 2
----------
Автор: contributor_id
Среднее время приготовления: minutes минут
```

    
Данные для создания строки получите из файлов `recipes_sample.csv` (__ЛР2__) и `steps_sample.xml` (__ЛР3__).
Вызовите данную функцию для рецепта с id `170895` и выведите (через `print`) полученную строку на экран.

In [ ]:
import xml.etree.ElementTree as ET

In [148]:
def show_info(name, steps, minutes, author_id):
    formatted_name = f'"{name.title()}"'

    steps_list = [f"{i+1}. {step.capitalize()}" for i, step in enumerate(steps)]
    steps_str = '\n'.join(steps_list)

    author_info = f"Автор: {author_id}"
    minutes_info = f"Среднее время приготовления: {minutes} минут"

    full_string = (
        f"{formatted_name}\n\n"
        f"{steps_str}\n"
        f"\n"
        f"{author_info}\n"
        f"{minutes_info}"
    )
    return full_string

In [149]:
recipes_df = pd.read_csv('recipes_sample.csv')
root =  ET.parse('steps_sample.xml').getroot()

steps_data = {}
for recipe_elem in root.findall('recipe'):
    recipe_id = int(recipe_elem.find('id').text)
    steps = [step.text for step in recipe_elem.find('steps').findall('step')]
    steps_data[recipe_id] = steps


recipe_info = recipes_df[recipes_df['id'] == 170895]
recipe_info = recipe_info.iloc[0]

name = recipe_info['name']
minutes = int(recipe_info['minutes'])
author_id = int(recipe_info['contributor_id'])
recipe_steps = steps_data.get(170895, [])
print(show_info(name, recipe_steps, minutes, author_id))

"Leeks And Parsnips  Sauteed Or Creamed"

1. Clean the leeks and discard the dark green portions
2. Cut the leeks lengthwise then into one-inch pieces
3. Melt the butter in a medium skillet , med
4. Heat
5. Add the garlic and fry 'til fragrant
6. Add leeks and fry until the leeks are tender , about 6-minutes
7. Meanwhile , peel and chunk the parsnips into one-inch pieces
8. Place in a steaming basket and steam 'til they are as tender as you prefer
9. I like them fork-tender
10. Drain parsnips and add to the skillet with the leeks
11. Add salt and pepper
12. Gently sautee together for 5-minutes
13. At this point you can serve it , or continue on and cream it:
14. In a jar with a screw top , add the half-n-half and arrowroot
15. Shake 'til blended
16. Turn heat to low under the leeks and parsnips
17. Pour in the arrowroot mixture , stirring gently as you pour
18. If too thick , gradually add the water
19. Let simmer for a couple of minutes
20. Taste to adjust seasoning , probably an addi

## Работа с регулярными выражениями

3\. Напишите регулярное выражение, которое ищет следующий паттерн в строке: число (1 цифра или более), затем пробел, затем слова: hour или hours или minute или minutes. Произведите поиск по данному регулярному выражению в каждом шаге рецепта с id 25082. Выведите на экран все непустые результаты, найденные по данному шаблону.

In [ ]:
for i, step in enumerate(steps_data.get(25082, [])):
    words = step.lower().replace(',', '').replace('.', '').replace('-', ' ').split()

    for j in range(len(words)):
        if words[j] in ['hour', 'hours', 'minute', 'minutes']:
            if j > 0 and words[j-1].isdigit():
                print(f"Step {i+1}: '{words[j-1]} {words[j]}'")

Step 6: '20 minutes'
Step 8: '10 minutes'
Step 10: '2 hours'
Step 14: '10 minutes'
Step 17: '20 minutes'
Step 17: '30 minutes'


4\. Напишите регулярное выражение, которое ищет шаблон вида "this..., but" _в начале строки_ . Между словом "this" и частью ", but" может находиться произвольное число букв, цифр, знаков подчеркивания и пробелов. Никаких других символов вместо многоточия быть не может. Пробел между запятой и словом "but" может присутствовать или отсутствовать.

Используя строковые методы `pd.Series`, выясните, для каких рецептов данный шаблон содержится в тексте описания. Выведите на экран количество таких рецептов и 3 примера подходящих описаний (текст описания должен быть виден на экране полностью).

In [ ]:
import re

In [ ]:
pattern = r"^this[a-zA-Z0-9_\s]*,\s?but"
matching_recipes = recipes[recipes['description'].fillna('').str.match(pattern, flags=re.IGNORECASE)]

print(f"len(matching_recipes): {len(matching_recipes)}")
print()

for index, row in matching_recipes.head(3).iterrows():
    print(f"- {row['description']}")

len(matching_recipes): 133

- this is a great meal eaten the same day ,but even better the next day , if you can wait! add your favourite spices, but try it first as it is and i think that you will enjoy the 'vegetable' taste. good for freezing.
- this was adapted from a recipe i found on the net, but i added julienne onion to the peppers.  this is a meal in itself, or you could have a small slice with a meat dish.  for those that like to have brunch, it's a bit different to your traditional quiche recipes.  if you love cheese, you could add 1/2 cup of your favorite to the egg mixture, then pour over peppers.
- this is kind of similar to some of the other versions out there, but it is the best and easiest i have found


5\. В текстах шагов рецептов обыкновенные дроби имеют вид "a / b". Используя регулярные выражения, уберите в тексте шагов рецепта с id 72367 пробелы до и после символа дроби. Выведите на экран шаги этого рецепта после их изменения.

In [ ]:
steps = []
pattern = r'(\d+)\s*/\s*(\d+)'

for step in steps_data.get(72367, []):
    step = re.sub(pattern, r'\1/\2', step)
    steps.append(step)
    print(f"- {step}")

- mix butter , flour , 1/3 c
- sugar and 1-1/4 t
- vanilla
- press into greased 9" springform pan
- mix cream cheese , 1/4 c
- sugar , eggs and 1/2 t
- vanilla beating until fluffy
- pour over dough
- combine apples , 1/3 c
- sugar and cinnamon
- arrange on top of cream cheese mixture and sprinkle with almonds
- bake at 350 for 45-55 minutes , or until tester comes out clean


### Сегментация текста

6\. Разбейте тексты шагов рецептов на слова при помощи пакета `nltk`. Посчитайте и выведите на экран кол-во уникальных слов среди всех рецептов. Словом называется любая последовательность алфавитных символов (для проверки можно воспользоваться `str.isalpha`). При подсчете количества уникальных слов не учитывайте регистр.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
unique_words = set()

for recipe_id, steps_list in steps_data.items():
    for step in steps_list:
        words = word_tokenize(step)
        for word in words:
            if word.isalpha():
                unique_words.add(word.lower())

print(f"len(unique_words): {len(unique_words)}")

len(unique_words): 14926


7\. Разбейте описания рецептов из `recipes` на предложения при помощи пакета `nltk`. Найдите 5 самых длинных описаний (по количеству _предложений_) рецептов в датасете и выведите строки фрейма, соответствующие этим рецептами, в порядке убывания длины.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
recipess= recipes.copy()
recipess['description'] = recipess['description'].fillna('')

recipess['sentences_count'] = recipess['description'].apply(lambda text: len(sent_tokenize(text)))

top = recipess.sort_values(by='sentences_count', ascending=False).head(5)
top[['name', 'description', 'sentences_count']]

,name,description,sentences_count
18408,my favorite buttercream icing for decorating,this wonderful icing is used for icing cakes a...,76
481,alligator claws avocado fritters with chipot...,a translucent golden-brown crust allows the gr...,27
22566,rich barley mushroom soup,this is one of the best soups i've ever made a...,24
6779,chocolate tea,i wrote this because there are an astounding l...,23
16296,little bunny foo foo cake carrot cake with c...,the first time i made this cake i grated a mil...,23


8\. Напишите функцию, которая для заданного предложения выводит информацию о частях речи слов, входящих в предложение, в следующем виде:
```
PRP   VBD   DT      NNS     CC   VBD      NNS        RB   
 I  omitted the raspberries and added strawberries instead
```
Для определения части речи слова можно воспользоваться `nltk.pos_tag`.

Проверьте работоспособность функции на названии рецепта с id 241106.

Обратите внимание, что часть речи должна находиться ровно посередине над соотвествующим словом, а между самими словами должен быть ровно один пробел.


In [ ]:
name = recipes[recipes['id'] == 241106]['name'].iloc[0]

words = word_tokenize(name)
pos_tags = nltk.pos_tag(words)

widths = [max(len(word), len(tag)) for word, tag in pos_tags]
tag_line = ' '.join(tag.center(width) for (word, tag), width in zip(pos_tags, widths))
word_line = ' '.join(word.center(width) for (word, tag), width in zip(pos_tags, widths))

print(f"{tag_line}\n{word_line}")

   JJ     NNS    IN     NNS    VBP    JJ    CC   JJ   NNS  
eggplant steaks with chickpeas feta cheese and black olives
